In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import os
import pandas as pd
from dotenv import load_dotenv
from mal_client import MALClient
from anime_data import AnimeDataClient
from anime_recommender import SimilarityRecommender

load_dotenv(PROJECT_ROOT / ".env")

client_id = os.getenv("CLIENT_ID")

In [3]:
anime_data_client = AnimeDataClient(client_id, cache_file=PROJECT_ROOT / "data" / "anime_cache.json")

In [4]:
anime_data = anime_data_client.get_cache()

Finding best numerical feature set

In [5]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

numeric_candidates = pd.DataFrame(anime_data.values())

statistics_df = pd.json_normalize(numeric_candidates["statistics"])
statistics_df = statistics_df.rename(columns={
    "num_list_users": "statistics_num_list_users",
    "status.watching": "watching",
    "status.completed": "completed",
    "status.on_hold": "on_hold",
    "status.dropped": "dropped",
    "status.plan_to_watch": "plan_to_watch",
})
statistics_df = statistics_df.apply(pd.to_numeric, errors="coerce")

numeric_candidates = pd.concat(
    [numeric_candidates.drop(columns=["statistics"]), statistics_df],
    axis=1,
)

numeric_columns = [
    "mean",
    "popularity",
    "statistics_num_list_users",
    "watching",
    "completed",
    "on_hold",
    "dropped",
    "plan_to_watch",
]

numeric_analysis_df = (
    numeric_candidates[numeric_columns]
    .apply(pd.to_numeric, errors="coerce")
    .dropna()
)

numeric_analysis_df.head()

,mean,popularity,statistics_num_list_users,watching,completed,on_hold,dropped,plan_to_watch
0,8.89,115,1342023,180331,515335,96469,44801,505087
1,8.12,1206,236729,14229,94939,9799,6573,111189
2,9.11,3,3704780,289840,2677659,122234,65660,549387
3,9.03,8,3210024,385809,2208233,153798,70581,391603
4,8.24,276,824887,56382,324394,37641,37026,369444


In [6]:
def calculate_vif(df):
    rows = []
    for target_col in df.columns:
        X = df.drop(columns=[target_col]).to_numpy(dtype=float)
        y = df[target_col].to_numpy(dtype=float)

        model = LinearRegression()
        model.fit(X, y)
        r_squared = model.score(X, y)

        vif = np.inf if np.isclose(1 - r_squared, 0) else 1 / (1 - r_squared)
        rows.append({
            "feature": target_col,
            "r_squared_from_other_features": r_squared,
            "vif": vif,
        })

    return pd.DataFrame(rows).sort_values("vif", ascending=False)

reduced_numeric_analysis_df = numeric_analysis_df.drop(
    columns=[
        "completed",
        "on_hold",
        "statistics_num_list_users",
        # "watching",
        "dropped",
        "plan_to_watch",
        "popularity",
        # "mean",
    ],
    errors="ignore",
)

vif_results = calculate_vif(reduced_numeric_analysis_df)
vif_results

,feature,r_squared_from_other_features,vif
0,mean,0.108404,1.121585
1,watching,0.108404,1.121585


In [7]:
import ast

user_data_df = pd.read_csv("../data/tuning_user_data_100.csv", index_col=0)

user_data = {
    username: {
        key: ast.literal_eval(value)
        for key, value in user_series.dropna().items()
    }
    for username, user_series in user_data_df.items()
}

user_len = {
    username: len(data["data"])
    for username, data in user_data.items()
}

len(user_data)

100

Build features

In [8]:
from anime_features import AnimeFeatureBuilder

builder = AnimeFeatureBuilder(
    anime_data,
    max_tfidf_features=3000,
    n_svd_components=50
)

anime_df_num = builder.build_num_features().set_index("id")
anime_genres_df = builder.build_genre_features().set_index("anime_id")
anime_studios_df = builder.build_studio_features().set_index("anime_id")
synopsis_tfidf, synopsis_features = builder.build_synopsis_features()
synopsis_svd_df = builder.apply_svd(synopsis_tfidf, synopsis_features)

anime_df_complete = pd.concat([
    anime_df_num,
    anime_genres_df,
    synopsis_svd_df,
], axis=1).dropna()

builder.svd_explained_variance

np.float64(0.12783040142102292)

In [9]:
anime_df_num_new = pd.DataFrame(anime_data.values())
anime_df_num_new = anime_df_num_new.drop(
    columns=[
        "main_picture",
        "title",
        "synopsis",
        "media_type",
        "status",
        "genres",
        "rating",
        "recommendations",
        "studios",
        "related_anime",
    ],
    errors="ignore",
)

statistics_df = pd.json_normalize(anime_df_num_new["statistics"])
statistics_df = statistics_df.rename(columns={
    "num_list_users": "statistics_num_list_users",
    "status.watching": "watching",
    "status.completed": "completed",
    "status.on_hold": "on_hold",
    "status.dropped": "dropped",
    "status.plan_to_watch": "plan_to_watch",
})
statistics_df = statistics_df.apply(pd.to_numeric, errors="coerce")

anime_df_num_new = pd.concat([anime_df_num_new.drop(columns=["statistics"]), statistics_df], axis=1)

anime_df_num_new = anime_df_num_new.drop(
    columns=[
        "completed",
        "on_hold",
        "statistics_num_list_users",
        # "watching",
        "dropped",
        "plan_to_watch",
        "num_list_users",
        "num_scoring_users",
        "num_episodes",
        "rank",
        # "popularity",
        # "mean",
    ],
    errors="ignore",).set_index("id").dropna()

anime_df_num_new

,mean,popularity,watching
id,,,
19,8.89,115,180331
1827,8.12,1206,14229
5114,9.11,3,289840
11061,9.03,8,385809
13125,8.24,276,56382
...,...,...,...
7334,6.39,3446,1531
61067,6.39,5899,2222
12967,6.27,1286,11435


Pick the feature set to use below

In [35]:
feature_set_options = {
    "complete_num_genres_synopsis_svd": pd.concat(
        [anime_df_num, anime_genres_df, synopsis_svd_df],
        axis=1,
    ),
    "reduced_num_genres_synopsis_svd": pd.concat(
        [anime_df_num_new, anime_genres_df, synopsis_svd_df],
        axis=1,
    ),
    "complete_num_genres_synopsis_svd_studios": pd.concat(
        [anime_df_num, anime_genres_df, synopsis_svd_df, anime_studios_df],
        axis=1,
    ),
    "complete_numeric_only": anime_df_num,
    "genres_only": anime_genres_df,
    "studios_only": anime_studios_df,
    "complete_num_genres": pd.concat([anime_df_num, anime_genres_df], axis=1),
    "complete_num_studios": pd.concat([anime_df_num, anime_studios_df], axis=1),
    "complete_num_synopsis_svd": pd.concat([anime_df_num, synopsis_svd_df], axis=1),
    "genres_synopsis_svd": pd.concat([anime_genres_df, synopsis_svd_df], axis=1),
    "genres_studios": pd.concat([anime_genres_df, anime_studios_df], axis=1),
    "complete_num_genres_studios": pd.concat(
        [anime_df_num, anime_genres_df, anime_studios_df],
        axis=1,
    ),
    "complete_num_synopsis_svd_studios": pd.concat(
        [anime_df_num, synopsis_svd_df, anime_studios_df],
        axis=1,
    ),
}

selected_feature_set_name = "complete_num_synopsis_svd_studios"

anime_df = feature_set_options[selected_feature_set_name].dropna()
selected_feature_columns = list(anime_df.columns)

print(
    f"Selected feature set: {selected_feature_set_name} "
    f"({anime_df.shape[0]} anime x {anime_df.shape[1]} features)"
)


Selected feature set: complete_num_synopsis_svd_studios (4859 anime x 53 features)


Convert each anime in df to vectors

In [36]:
recommender = SimilarityRecommender()
anime_vectors = recommender.create_anime_vectors(anime_df)
anime_df_scaled = recommender.anime_df_scaled

anime_df_scaled.head()

,mean,popularity,watching,synopsis_svd_0,synopsis_svd_1,synopsis_svd_2,synopsis_svd_3,synopsis_svd_4,synopsis_svd_5,synopsis_svd_6,...,synopsis_svd_40,synopsis_svd_41,synopsis_svd_42,synopsis_svd_43,synopsis_svd_44,synopsis_svd_45,synopsis_svd_46,synopsis_svd_47,synopsis_svd_48,synopsis_svd_49
19,2.539735,-1.089806,4.201176,0.110086,-0.015490,-0.021589,-0.022465,-0.018593,-0.000629,0.020175,...,0.016943,0.032931,0.012431,-0.014967,-0.019022,0.018090,-0.023762,0.012277,-0.016603,0.015735
1827,1.528637,-0.860243,0.007756,0.153552,-0.022572,-0.082775,-0.084264,-0.020766,0.057609,0.011190,...,-0.070425,0.025268,-0.025372,-0.058050,0.036880,0.049105,0.007465,0.037841,0.020525,-0.029334
5114,2.828620,-1.113372,6.965845,0.150778,-0.021285,-0.048890,-0.055630,-0.000358,0.031683,-0.009206,...,0.019706,-0.001803,0.002828,0.030142,-0.013082,-0.036704,0.010913,-0.036621,0.035189,-0.027492
11061,2.723571,-1.112320,9.388684,0.138365,-0.020031,0.001671,-0.045469,-0.021396,0.027681,-0.030591,...,-0.040826,-0.018177,-0.036637,-0.048338,0.026112,-0.016480,0.021614,0.043853,-0.037449,-0.011149
13125,1.686211,-1.055929,1.071953,0.212720,-0.030132,0.015872,-0.048893,-0.016133,0.026478,0.019985,...,-0.028060,-0.003472,-0.049890,-0.001796,-0.003887,-0.026461,-0.037747,0.014583,-0.004348,0.039909


In [33]:
eval_users = ['LoyJoy', 'DanteNani2004', 'Kutu_Mastor', 'Dadalt',
 'yorkie_pro', 'Opelo_Stradyon', 'Captn_Cook', 'Dinterdos123', 
 'Ebelha', 'Moonshyla', 'LocNguyen',
 'IM_C4', 'HumanRat', 'KSTTK', 'DaiShi57', 'dareggon',
 'pensa89', 'MarnikBe', 'Edison-great',
 'TheRealist68', 'MeloniaShaby', 'Irseus', 'AkiAki_Akira',
 'GrumbleDango', 'Vurnox', 'xF0x', 'alfalfy', 'Siipn',
 'Haileytokar']

Tune feature set seletion for 29 users

In [34]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import BayesianRidge
from sklearn.neighbors import KNeighborsRegressor

from anime_evaluation import RankingMetricEvaluator


n_runs = 100
result_top_ks = (5,)
uncertainty_weight = 15
clip_predictions = False
random_state = 42

if "selected_feature_set_name" not in globals():
    selected_feature_set_name = "manual_anime_df"

if "selected_feature_columns" not in globals():
    selected_feature_columns = list(anime_df.columns)

eval_users = [user.strip() for user in eval_users]

missing_users = [
    user for user in eval_users
    if user not in user_data
]
if missing_users:
    raise KeyError(
        f"These eval users are missing from user_data: {missing_users}"
    )

score_client = MALClient(client_id)

feature_selection_user_scores = {
    user: score_client.get_scores(user_data[user])
    for user in eval_users
}

split_seeds = np.random.default_rng(random_state).integers(
    0,
    2**32 - 1,
    size=n_runs,
    dtype=np.uint32,
)

all_run_rows = []
skipped_users = []

for i, (username, scores) in enumerate(
    feature_selection_user_scores.items(),
    start=1,
):
    print(
        f"[{i}/{len(feature_selection_user_scores)}] "
        f"username: {username}"
    )

    try:
        ranking_evaluator = RankingMetricEvaluator(
            anime_df_scaled=anime_df_scaled,
            anime_df=anime_df,
            scores=scores,
            anime_data_client=anime_data_client,
            anime_data=anime_data,
            builder=builder,
            recommender=recommender,
            heldout_fraction=0.25,
        )
    except ValueError as error:
        print(f"Skipping {username}: {error}")
        skipped_users.append({
            "username": username,
            "error": str(error),
        })
        continue

    for run, split_seed in enumerate(split_seeds, start=1):
        try:
            train_eval, train_ids, candidate_ids, test_eval = (
                ranking_evaluator._sample_ranking_split(
                    random_state=int(split_seed)
                )
            )
        except ValueError as error:
            print(f"Skipping {username}, run {run}: {error}")
            continue

        if not train_ids or not candidate_ids or test_eval.empty:
            continue

        X_train = (
            ranking_evaluator.anime_df_scaled
            .loc[train_ids]
            .to_numpy()
        )
        y_train = train_eval["score"].to_numpy(dtype=float)

        X_candidates = (
            ranking_evaluator.anime_df_scaled
            .loc[candidate_ids]
            .to_numpy()
        )

        relevance_by_id = (
            test_eval
            .set_index("anime_id")["relevance"]
            .to_dict()
        )

        base_recommendations = pd.DataFrame({
            "anime_id": candidate_ids,
        })

        base_recommendations["relevance"] = (
            base_recommendations["anime_id"]
            .map(relevance_by_id)
            .fillna(0)
            .astype(int)
        )

        bayesian_model = BayesianRidge()
        bayesian_model.fit(X_train, y_train)

        bayesian_predictions, bayesian_uncertainty = (
            bayesian_model.predict(
                X_candidates,
                return_std=True,
            )
        )

        bayesian_predictions = (
            ranking_evaluator._prepare_prediction_scores(
                bayesian_predictions,
                clip_predictions=clip_predictions,
            )
        )

        bayesian_uncertainty = (
            ranking_evaluator._scale_uncertainty(
                bayesian_uncertainty
            )
        )

        bayesian_recommendations = base_recommendations.copy()
        bayesian_recommendations["ranking_score"] = (
            bayesian_predictions
            - uncertainty_weight * bayesian_uncertainty
        )

        bayesian_recommendations = (
            bayesian_recommendations.sort_values(
                ["ranking_score", "anime_id"],
                ascending=[False, True],
                kind="mergesort",
            )
        )

        knn_neighbors = len(train_ids)

        knn_model = KNeighborsRegressor(
            n_neighbors=knn_neighbors,
            weights="distance",
            metric="cosine",
            n_jobs=-1,
        )
        knn_model.fit(X_train, y_train)

        knn_predictions = (
            ranking_evaluator._prepare_prediction_scores(
                knn_model.predict(X_candidates),
                clip_predictions=False,
            )
        )

        knn_recommendations = base_recommendations.copy()
        knn_recommendations["ranking_score"] = knn_predictions

        knn_recommendations = knn_recommendations.sort_values(
            ["ranking_score", "anime_id"],
            ascending=[False, True],
            kind="mergesort",
        )

        ranked_models = {
            "bayesian_ridge": bayesian_recommendations,
            "knn_all_available": knn_recommendations,
        }

        if (
            ranking_evaluator.anime_df is not None
            and "mean" in ranking_evaluator.anime_df.columns
        ):
            global_recommendations = base_recommendations.copy()

            global_recommendations["global_mean"] = (
                ranking_evaluator.anime_df["mean"]
                .reindex(global_recommendations["anime_id"])
                .to_numpy()
            )

            if (
                "num_scoring_users"
                in ranking_evaluator.anime_df.columns
            ):
                global_recommendations["num_scoring_users"] = (
                    ranking_evaluator.anime_df["num_scoring_users"]
                    .reindex(global_recommendations["anime_id"])
                    .to_numpy()
                )

            global_recommendations = (
                ranking_evaluator._sort_with_anime_tiebreakers(
                    global_recommendations,
                    "global_mean",
                )
            )

            ranked_models["global_mean"] = global_recommendations

        for model_name, recommendations in ranked_models.items():
            metric_rows = (
                ranking_evaluator._score_ranking_top_k(
                    recommendations,
                    result_top_ks,
                    test_eval,
                )
            )

            for row in metric_rows:
                row["username"] = username
                row["model"] = model_name
                row["run"] = run
                row["split_seed"] = int(split_seed)
                row["score_count"] = len(scores)
                row["train_count"] = len(train_ids)
                row["candidate_count"] = len(candidate_ids)
                row["knn_neighbors"] = (
                    knn_neighbors
                    if model_name == "knn_all_available"
                    else np.nan
                )
                all_run_rows.append(row)

if not all_run_rows:
    raise ValueError(
        "No users had enough usable ratings for evaluation."
    )

feature_selection_run_results = pd.DataFrame(all_run_rows)

feature_selection_per_user_results_df = (
    RankingMetricEvaluator.summarize_ranking(
        feature_selection_run_results,
        ["username", "model", "k"],
    )
)

per_user_metadata = (
    feature_selection_run_results
    .groupby(
        ["username", "model", "k"],
        as_index=False,
    )
    .agg(
        valid_runs=("run", "nunique"),
        score_count=("score_count", "first"),
        avg_train_count=("train_count", "mean"),
        avg_candidate_count=("candidate_count", "mean"),
        avg_knn_neighbors=("knn_neighbors", "mean"),
    )
)

feature_selection_per_user_results_df = (
    feature_selection_per_user_results_df
    .merge(
        per_user_metadata,
        on=["username", "model", "k"],
        how="left",
    )
)

feature_selection_per_user_results_df["feature_set"] = (
    selected_feature_set_name
)
feature_selection_per_user_results_df["feature_columns"] = (
    ",".join(selected_feature_columns)
)
feature_selection_per_user_results_df["n_features"] = (
    len(selected_feature_columns)
)
feature_selection_per_user_results_df["feature_rows"] = (
    len(anime_df)
)
feature_selection_per_user_results_df[
    "bayesian_uncertainty_weight"
] = uncertainty_weight
feature_selection_per_user_results_df[
    "clip_predictions"
] = clip_predictions
feature_selection_per_user_results_df["n_runs"] = n_runs
feature_selection_per_user_results_df[
    "random_state"
] = random_state
feature_selection_per_user_results_df[
    "svd_explained_variance"
] = getattr(
    builder,
    "svd_explained_variance",
    np.nan,
)

feature_selection_summary = (
    feature_selection_per_user_results_df
    .groupby(
        [
            "feature_set",
            "model",
            "bayesian_uncertainty_weight",
            "clip_predictions",
            "n_runs",
            "k",
        ],
        as_index=False,
    )
    .agg(
        avg_ndcg_at_k=("avg_ndcg_at_k", "mean"),
        std_ndcg_at_k_across_users=(
            "avg_ndcg_at_k",
            "std",
        ),
        avg_precision_at_k=(
            "avg_precision_at_k",
            "mean",
        ),
        std_precision_at_k_across_users=(
            "avg_precision_at_k",
            "std",
        ),
        avg_mrr_at_k=("avg_mrr_at_k", "mean"),
        std_mrr_at_k_across_users=(
            "avg_mrr_at_k",
            "std",
        ),
        mean_within_user_std_ndcg_at_k=(
            "std_ndcg_at_k",
            "mean",
        ),
        mean_within_user_std_precision_at_k=(
            "std_precision_at_k",
            "mean",
        ),
        mean_within_user_std_mrr_at_k=(
            "std_mrr_at_k",
            "mean",
        ),
        avg_relevant_hits_at_k=(
            "avg_relevant_hits_at_k",
            "mean",
        ),
        avg_strong_hits_at_k=(
            "avg_strong_hits_at_k",
            "mean",
        ),
        avg_test_relevant=(
            "avg_test_relevant",
            "mean",
        ),
        avg_test_strong_relevant=(
            "avg_test_strong_relevant",
            "mean",
        ),
        users_evaluated=("username", "nunique"),
        avg_score_count=("score_count", "mean"),
        avg_train_count=("avg_train_count", "mean"),
        avg_candidate_count=("avg_candidate_count", "mean"),
        avg_knn_neighbors=("avg_knn_neighbors", "mean"),
    )
)

feature_selection_summary["feature_columns"] = (
    ",".join(selected_feature_columns)
)
feature_selection_summary["n_features"] = (
    len(selected_feature_columns)
)
feature_selection_summary["feature_rows"] = len(anime_df)
feature_selection_summary["random_state"] = random_state
feature_selection_summary["svd_explained_variance"] = getattr(
    builder,
    "svd_explained_variance",
    np.nan,
)

feature_selection_summary = (
    feature_selection_summary
    .sort_values(
        [
            "k",
            "avg_precision_at_k",
            "avg_ndcg_at_k",
            "model",
        ],
        ascending=[True, False, False, True],
    )
    .reset_index(drop=True)
)

METRICS_DIR = (
    PROJECT_ROOT
    / "metrics"
    / "multi_user_metrics"
)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

summary_path = (
    METRICS_DIR
    / "multi_user_feature_set_selection_results.csv"
)

per_user_path = (
    METRICS_DIR
    / "multi_user_feature_set_selection_per_user_results.csv"
)

try:
    existing_summary = pd.read_csv(summary_path)
except (FileNotFoundError, pd.errors.EmptyDataError):
    existing_summary = pd.DataFrame()

updated_summary = pd.concat(
    [existing_summary, feature_selection_summary],
    ignore_index=True,
)

updated_summary = updated_summary.drop_duplicates(
    subset=[
        "feature_set",
        "model",
        "bayesian_uncertainty_weight",
        "clip_predictions",
        "n_runs",
        "k",
    ],
    keep="last",
)

updated_summary.to_csv(
    summary_path,
    index=False,
)

try:
    existing_per_user = pd.read_csv(per_user_path)
except (FileNotFoundError, pd.errors.EmptyDataError):
    existing_per_user = pd.DataFrame()

updated_per_user = pd.concat(
    [
        existing_per_user,
        feature_selection_per_user_results_df,
    ],
    ignore_index=True,
)

updated_per_user = updated_per_user.drop_duplicates(
    subset=[
        "feature_set",
        "model",
        "bayesian_uncertainty_weight",
        "clip_predictions",
        "n_runs",
        "k",
        "username",
    ],
    keep="last",
)

updated_per_user.to_csv(
    per_user_path,
    index=False,
)

if skipped_users:
    skipped_path = (
        METRICS_DIR
        / "multi_user_feature_set_selection_skipped_users.csv"
    )
    pd.DataFrame(skipped_users).to_csv(
        skipped_path,
        index=False,
    )

print(f"Wrote summary rows to {summary_path}")
print(f"Wrote per-user rows to {per_user_path}")

display(feature_selection_summary)

[1/29] username: LoyJoy
[2/29] username: DanteNani2004
[3/29] username: Kutu_Mastor
[4/29] username: Dadalt
[5/29] username: yorkie_pro
[6/29] username: Opelo_Stradyon
[7/29] username: Captn_Cook
[8/29] username: Dinterdos123
[9/29] username: Ebelha
[10/29] username: Moonshyla
[11/29] username: LocNguyen
Fetching details for 11 anime...
All requested anime were already cached.
[12/29] username: IM_C4
[13/29] username: HumanRat
[14/29] username: KSTTK
[15/29] username: DaiShi57
[16/29] username: dareggon
[17/29] username: pensa89
[18/29] username: MarnikBe
[19/29] username: Edison-great
[20/29] username: TheRealist68
[21/29] username: MeloniaShaby
[22/29] username: Irseus
[23/29] username: AkiAki_Akira
[24/29] username: GrumbleDango
[25/29] username: Vurnox
[26/29] username: xF0x
[27/29] username: alfalfy
[28/29] username: Siipn
[29/29] username: Haileytokar
Wrote summary rows to c:\ML\anirec_v2\metrics\multi_user_metrics\multi_user_feature_set_selection_results.csv
Wrote per-user rows 

,feature_set,model,bayesian_uncertainty_weight,clip_predictions,n_runs,k,avg_ndcg_at_k,std_ndcg_at_k_across_users,avg_precision_at_k,std_precision_at_k_across_users,...,users_evaluated,avg_score_count,avg_train_count,avg_candidate_count,avg_knn_neighbors,feature_columns,n_features,feature_rows,random_state,svd_explained_variance
0,complete_num_genres_synopsis_svd,knn_all_available,15,False,100,5,0.424674,0.194446,0.438483,0.196334,...,29,251.965517,188.655172,4670.344828,188.655172,"mean,popularity,watching,Action,Adult Cast,Adv...",130,4859,42,0.12783
1,complete_num_genres_synopsis_svd,bayesian_ridge,15,False,100,5,0.187790,0.162357,0.210414,0.173749,...,29,251.965517,188.655172,4670.344828,NaN,"mean,popularity,watching,Action,Adult Cast,Adv...",130,4859,42,0.12783
2,complete_num_genres_synopsis_svd,global_mean,15,False,100,5,0.129242,0.105079,0.140966,0.126220,...,29,251.965517,188.655172,4670.344828,NaN,"mean,popularity,watching,Action,Adult Cast,Adv...",130,4859,42,0.12783


## Results

Stick with Numeric + Genres features only (complete_num_genres). The synopsis SVD version improved KNN Precision@5 only from 0.43793 to 0.43848, a difference of about 0.00055. Its NDCG improvement was larger, about 0.0069, but since Precision@5 is your primary metric, the additional 50 synopsis features do not provide enough benefit to justify the complexity. The numeric-plus-genres representation is the better final choice.
